In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # PyTorch Image Models (ResNeSt için şart)
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pandas as pd
import numpy as np

# Uyarıları gizlemek için (isteğe bağlı)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# --- KONFIGURASYON (ConvNeXt Base + Aug) ---
# Tiny/small yerine base kullaniyoruz.
MODEL_NAME = 'convnext_base'
EXPERIMENT_NAME = "ConvNeXt_Base_Aug_R1"

# Hiperparametreler
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 0.0002
NUM_CLASSES = 8
DROPOUT_RATE = 0.3
WEIGHT_DECAY = 0.0007
LABEL_SMOOTHING = 0.08
USE_CLASS_WEIGHTED_LOSS = True
EARLY_STOPPING_PATIENCE = 10

# Calistirma etiketleri (baseline ciktilariyla karismamasi icin)
dropout_tag = f"d{int(round(DROPOUT_RATE * 10)):02d}"
ls_tag = f"ls{int(round(LABEL_SMOOTHING * 100)):02d}"
wd_tag = f"wd{WEIGHT_DECAY:.0e}".replace("+", "")
lr_tag = f"lr{LEARNING_RATE:.0e}".replace("+", "")
cwl_tag = "cwlON" if USE_CLASS_WEIGHTED_LOSS else "cwlOFF"
RUN_TAG = f"{dropout_tag}_{ls_tag}_{wd_tag}_b{BATCH_SIZE}_e{EPOCHS}_{lr_tag}_aug_{cwl_tag}"
RUN_NAME = f"{EXPERIMENT_NAME}_{RUN_TAG}"

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "prepared-data")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models", "pytorch", RUN_NAME)
RESULT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME)
PLOTS_DIR = os.path.join(RESULT_OUTPUT_DIR, "plots")
REPORTS_DIR = os.path.join(RESULT_OUTPUT_DIR, "reports")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, f"best_{MODEL_NAME}.pth")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Run Tag: {RUN_TAG}")
print(f"Veri Yolu: {DATA_DIR}")
print(f"Kayit Yeri: {OUTPUT_DIR}")
print(f"Best Model Yolu: {BEST_MODEL_PATH}")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"Label Smoothing: {LABEL_SMOOTHING}")
print(f"Class Weighted Loss: {USE_CLASS_WEIGHTED_LOSS}")
print(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")


In [ ]:
# ImageNet Normalize Degerleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri Donusumleri (egitimde augmentation acik)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri Olustur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderlari Olustur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x == 'train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

# Class-weighted loss icin egitim seti sinif agirliklari
train_targets = np.array(image_datasets['train'].targets)
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights = class_counts.sum() / (NUM_CLASSES * np.maximum(class_counts, 1))
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32)

print(f"Siniflar: {class_names}")
print(f"Sinif Sayilari: {class_counts.tolist()}")
print(f"Sinif Agirliklari: {[round(w, 4) for w in class_weights.tolist()]}")
print(f"Egitim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")


In [ ]:
def create_model():
    print(f"Model indiriliyor: {MODEL_NAME}...")
    # pretrained=True ile ImageNet agirliklarini aliyoruz
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE)
    return model

model = create_model()
model = model.to(DEVICE)

# Kayip Fonksiyonu ve Optimizer
loss_weight = CLASS_WEIGHTS_TENSOR.to(DEVICE) if USE_CLASS_WEIGHTED_LOSS else None
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=loss_weight)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning Rate Scheduler (Plato gorulurse ogrenme hizini dusur)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

print("Model GPU'ya yuklendi ve egitime hazir.")


In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    required_globals = [
        "dataloaders", "dataset_sizes", "BEST_MODEL_PATH", "DEVICE", "EARLY_STOPPING_PATIENCE"
    ]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        raise RuntimeError(
            f"Eksik degisken(ler): {missing}. Lutfen once veri ve konfigurasyon hucrelerini calistirin."
        )

    os.makedirs(os.path.dirname(BEST_MODEL_PATH), exist_ok=True)

    since = time.time()
    best_acc = 0.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch dongusu
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Gecmisi kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val':
                scheduler.step(epoch_loss)
                current_lr = optimizer.param_groups[0]['lr']
                history['lr'].append(current_lr)
                print(f"Guncel LR: {current_lr:.8f}")

                if epoch_acc > best_acc:
                    best_acc = epoch_acc

                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    epochs_without_improvement = 0
                    with open(BEST_MODEL_PATH, 'wb') as f:
                        torch.save(model.state_dict(), f)
                    print(f"En Iyi Model (val loss: {best_val_loss:.4f}) -> {BEST_MODEL_PATH}")
                else:
                    epochs_without_improvement += 1
                    print(
                        f"Val loss iyilesmedi: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
                    )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Erken durdurma tetiklendi.")
            break

    time_elapsed = time.time() - since
    print(f'\nEgitim Tamamlandi: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En Iyi Validasyon Dogrulugu: {best_acc:.4f}')
    print(f'En Dusuk Validasyon Kaybi: {best_val_loss:.4f}')

    # En iyi agirliklari geri yukle
    with open(BEST_MODEL_PATH, 'rb') as f:
        model.load_state_dict(torch.load(f, map_location=DEVICE))
    return model, history


In [ ]:
# --- BASLAT ---
required_vars = ["model", "criterion", "optimizer", "scheduler", "train_model"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        f"Eksik degisken(ler): {missing}. Lutfen once model/egitim hazirlik hucrelerini calistirin."
    )
num_epochs = int(globals().get("EPOCHS", 50))
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=num_epochs)


In [ ]:
plt.figure(figsize=(14, 5))

# Dogruluk grafigi
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title(f'{MODEL_NAME} Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Kayip grafigi
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Kaydet ve goster
plot_path = os.path.join(PLOTS_DIR, f'training_graph_{MODEL_NAME}_{RUN_TAG}.png')
plt.savefig(plot_path)
plt.show()
print(f"Grafikler kaydedildi: {plot_path}")


In [ ]:
print("\nTEST SETI DEGERLENDIRMESI")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 1. Classification Report (F1, Recall, Precision)
print("\nSiniflandirma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

report_path = os.path.join(REPORTS_DIR, f'classification_report_{MODEL_NAME}_{RUN_TAG}.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"Rapor kaydedildi: {report_path}")

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gercek')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plot_path = os.path.join(PLOTS_DIR, f'confusion_matrix_{MODEL_NAME}_{RUN_TAG}.png')
plt.savefig(plot_path)
plt.show()
print(f"Confusion matrix kaydedildi: {plot_path}")
